# Web Scraping Falcon 9 Launch Records from Wikipedia

Requests the Wikipedia page, parses the launch tables with BeautifulSoup and builds a DataFrame.

> **Note:** needs internet access. Wikipedia layouts change; use a fixed revision URL if the live page breaks the parser.

In [ ]:
import requests, re, pandas as pd
from bs4 import BeautifulSoup
URL="https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
html=requests.get(URL,headers={"User-Agent":"Mozilla/5.0"}).text
soup=BeautifulSoup(html,"html.parser"); soup.title.string

In [ ]:
tables=soup.find_all("table","wikitable plainrowheaders collapsible")
first=tables[0]
column_names=[]
for th in first.find_all("th"):
    name="".join(th.stripped_strings)
    if name and name.isalpha()==False and len(name)>0: pass
    if name: column_names.append(name)
column_names=[c for c in column_names if c][:11]
column_names

In [ ]:
launch_dict={k:[] for k in ["Flight No.","Date","Time","Booster version","Launch site","Payload","Payload mass","Orbit","Customer","Launch outcome","Booster landing"]}
for table in tables:
    for row in table.find_all("tr"):
        cells=row.find_all(["th","td"])
        if len(cells)<9: continue
        txt=[" ".join(c.stripped_strings) for c in cells]
        if not re.match(r"^\d+$",txt[0]): continue
        for k,v in zip(launch_dict.keys(),txt[:11]): launch_dict[k].append(v)
df=pd.DataFrame({k:pd.Series(v) for k,v in launch_dict.items()})
df.shape

In [ ]:
df.to_csv("data/spacex_web_scraped.csv",index=False)
df.head()